# w_inst calibration helper

Pick production constraints for the BH fitter (`w_inst_default`, `w_inst_bounds`, optionally `fix_w_inst`, plus `dx_tol_nm` and `base_bound`) by **inspecting** the distribution of free-fit parameters over a small set of representative spectra.

**This notebook is intentionally manual.**  It does not auto-pick bounds.  You decide whether the histograms support a tighter constraint or a fixed value.

Procedure:

1. Pick a few representative shots (`SHOTS`) and an explicit (small) set of `FRAMES` / `CHANNELS`.
2. Run `run_bh_batch` with **loose** bounds (the defaults).
3. Use `summarize_fit_distribution` + `plot_fit_distributions` to inspect `w_inst`, `dx`, `base`.
4. Optionally overlay candidate bounds on the histograms.
5. Adopt the chosen values explicitly in `examples/12_batch_batch.ipynb` (`W_INST_DEFAULT`, `W_INST_BOUNDS`, ...) or in your YAML config.

In [ ]:
from pathlib import Path

DATA_DIR = Path("~/Dropbox/Experiments/2025-LHD-BH/133mORCA").expanduser()
OUT_DIR = Path("bh_calibration_results")

SHOTS = [193788]
FRAMES = [6, 7, 8, 9, 10]
CHANNELS = [3, 5, 10, 20, 28]

BACKGROUND_FRAMES = (0, 1, 2, 3)
BH_FIT_WAVELENGTH_RANGE_NM = (433.05, 433.90)
BH_SCALE_WAVELENGTH_RANGE_NM = (433.08, 433.30)

CW_NM = 431.91
SCALE = 1.0
TIME_RANGE = (0.0, 10.0)

# Loose w_inst bounds for the calibration scan.  Keep generous; we want
# to see where the optimizer naturally lands.
LOOSE_W_INST_BOUNDS = (0.005, 0.06)

# Optional R2 quality cut when summarizing.
R2_MIN = 0.5

# Candidate bounds you want to overlay on the histograms (or None).
CANDIDATE_BOUNDS = {
    "w_inst": (0.020, 0.024),
    "dx": (-0.10, 0.10),
    "base": (-0.03, 0.03),
}

In [ ]:
from bh_molecule import run_folder_batch
from bh_molecule.workflows.calibration import (
    summarize_fit_distribution,
    plot_fit_distributions,
)

out_dir = OUT_DIR if OUT_DIR.is_absolute() else (Path.cwd() / OUT_DIR).resolve()
print(f"Output folder:           {out_dir}")
print(f"Shots:                   {SHOTS}")
print(f"Frames:                  {FRAMES}")
print(f"Channels:                {CHANNELS}")
print(f"Background frames:       {BACKGROUND_FRAMES}")
print(f"BH fit window:           {BH_FIT_WAVELENGTH_RANGE_NM} nm")
print(f"BH scale window:         {BH_SCALE_WAVELENGTH_RANGE_NM} nm")
print(f"Loose w_inst bounds:     {LOOSE_W_INST_BOUNDS} nm")

results = run_folder_batch(
    DATA_DIR,
    frames=FRAMES,
    channels=CHANNELS,
    shots=SHOTS,
    cw=CW_NM,
    scale=SCALE,
    time_range=TIME_RANGE,
    background_frames=BACKGROUND_FRAMES,
    bh_fit_range=BH_FIT_WAVELENGTH_RANGE_NM,
    bh_scale_range=BH_SCALE_WAVELENGTH_RANGE_NM,
    w_inst_bounds=LOOSE_W_INST_BOUNDS,
    fix_w_inst=False,
    out_dir=out_dir,
    save_frames=False,
)
list(results.keys())

In [ ]:
import pandas as pd

frames_list = []
for shot_id, df_shot in results.items():
    if df_shot is None or df_shot.empty:
        continue
    d = df_shot.copy()
    d["shot"] = shot_id
    frames_list.append(d)

if frames_list:
    df = pd.concat(frames_list, ignore_index=True)
else:
    # Fallback: read the on-disk CSVs (resume path).
    dfs = []
    for shot in SHOTS:
        csv = out_dir / str(shot) / "summary.csv"
        if csv.is_file():
            d = pd.read_csv(csv)
            d["shot"] = str(shot)
            dfs.append(d)
    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Total fits: {len(df)}")
df.head()

In [ ]:
summary = summarize_fit_distribution(
    df, params=("w_inst", "dx", "base"), r2_min=R2_MIN
)
print(f"--- summary (R2 >= {R2_MIN}) ---")
summary

In [ ]:
fig, axes = plot_fit_distributions(
    df,
    params=("w_inst", "dx", "base"),
    suggested_bounds=CANDIDATE_BOUNDS,
    bins=25,
    r2_min=R2_MIN,
)
fig

## Adopt production constraints

Inspect the summary table and histograms above.  Reasonable production rules:

- If `w_inst` clusters tightly around a single value across frames/channels with low MAD, consider either:
  - tight bounds: `w_inst_bounds = (p16, p84)` or a slightly wider hand-picked interval, **or**
  - a fixed value: `fix_w_inst = True` with `w_inst_default = median`.
- If `dx` stays well inside a narrow window, lower `dx_tol_nm` accordingly (but keep some slack for CW drift).
- If `base` already sits near zero, the default `base_bound = 0.03` is usually fine.

Then copy the chosen values into `examples/12_batch_batch.ipynb` (the `W_INST_*`, `DX_TOL_NM`, `BASE_BOUND` cell) or into your YAML config.

**Do not** push automatic bound selection: keep the choice explicit and reviewable.